In [29]:
import fitz
import os, math
import pandas as pd
import numpy as np
from pathlib import Path
from utils import Helper
utils = Helper()

out_dir = r"C:\Users\kaustubh.keny\Projects\OUTPUTS"

In [ ]:
file_path = r"Q1 2026 FILES\Quarterly Results 2026 Q1_TYPE.xlsx"
df = pd.read_excel(file_path)

print(f"TOTAL PDFS: {df.shape}")

def max_consecutive_scanned(x):
    max_run = 0
    current_run = 0

    for v in x:
        if v == "scanned":
            current_run += 1
            max_run = max(max_run, current_run)
        else:
            current_run = 0

    return max_run

cons_scan = (
    df.groupby("pdf_name")["type"]
      .apply(max_consecutive_scanned)
      .rename("max_consecutive_scanned")
)

summary = (
    df.groupby("pdf_name")["type"]
      .agg(
          total_pages="count",
          scanned_pages=lambda x: (x == "scanned").sum()
      )
)

summary["scanned_ratio"] = (
    summary["scanned_pages"] / summary["total_pages"]
)

summary["PDF_TYPE_33"] = np.where(
    summary["scanned_ratio"] >= 0.33,
    "SCANNED",
    "TEXT"
)

summary["PDF_TYPE_25"] = np.where(
    summary["scanned_ratio"] >= 0.25,
    "SCANNED",
    "TEXT"
)

summary.join(cons_scan).reset_index().to_excel("RATIO_PDF.xlsx", index=False)

TOTAL PDFS: (34749, 3)


In [10]:
d1 = summary[(summary["PDF_TYPE_25"] == "TEXT") & ( summary["total_pages"] > 1)]
d2 = summary[(summary["PDF_TYPE_25"] == "SCANNED") & ( summary["total_pages"] > 1)]

In [19]:
text_files = d1.index.to_list()
print(len(text_files), text_files[:10])

2785 ['20 Microns Ltd..pdf', '360 One Wam Ltd..pdf', '3B Blackbio Dx Ltd..pdf', '3M India Ltd..pdf', '3P Land Holdings Ltd..pdf', '5Paisa Capital Ltd..pdf', '63 Moons Technologies Ltd..pdf', '7NR Retail Ltd..pdf', 'A B Cotspin India Ltd..pdf', 'A B Infrabuild Ltd..pdf']


In [ ]:
hit_path = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\eda_pdf\Q1 2026 FILES\Quarterly Results 2026 Q1_HITS.xlsx"
hit_df = pd.read_excel(hit_path)


In [32]:
summary = hit_df[ (hit_df.numeric_count>2) & (hit_df.total_x_hits > 0)]
df = summary[["pdf","page_n"]]

In [ ]:
import os
import shutil

folder_path = r"C:\Users\kaustubh.keny\Documents\Quarterly Results 2026 Q1"

batch_limit_mb = 100
batch = 1
size_count = 0

output_folder = os.path.join(out_dir, f"batch_{batch}")
os.makedirs(output_folder, exist_ok=True)

for filename in text_files:
    fp = os.path.join(folder_path, filename)
    size = os.path.getsize(fp) / (1024 * 1024)  # MB

    # Create a new batch if this file would exceed the limit
    if size_count + size > batch_limit_mb:
        batch += 1
        size_count = 0

        output_folder = os.path.join(out_dir, f"batch_{batch}")
        os.makedirs(output_folder, exist_ok=True)

    shutil.copy(fp, output_folder)
    size_count += size

In [36]:
import os
import math
import fitz  # PyMuPDF
import pandas as pd

pdf_folder = r"C:\Users\kaustubh.keny\Documents\Quarterly Results 2026 Q1"
output_folder = os.path.join(out_dir,"batch_data")

os.makedirs(output_folder, exist_ok=True)

# df columns:
# pdf_name
# page_number

df = df.copy()

batch_size = 10
df["batch_pdf_name"] = None

num_batches = math.ceil(len(df) / batch_size)

for batch_no in range(num_batches):

    start_idx = batch_no * batch_size
    end_idx = min(start_idx + batch_size, len(df))

    batch_df = df.iloc[start_idx:end_idx]

    batch_pdf_name = f"batch_{batch_no + 1}.pdf"
    batch_pdf_path = os.path.join(output_folder, batch_pdf_name)

    batch_doc = fitz.open()

    for idx in batch_df.index:

        pdf_name = df.loc[idx, "pdf"]
        page_number = int(df.loc[idx, "page_n"])

        src_pdf_path = os.path.join(pdf_folder, f"{pdf_name}.pdf")

        src_doc = fitz.open(src_pdf_path)

        # convert to zero-based page index
        src_page_idx = page_number - 1

        # copy page
        batch_doc.insert_pdf(
            src_doc,
            from_page=src_page_idx,
            to_page=src_page_idx
        )

        # get newly added page
        page = batch_doc[-1]

        # add label at top-left
        page.insert_text(
            (20, 30),
            f"{pdf_name} | Page {page_number}",
            fontsize=20,
            color=(0, 0, 0)
        )

        src_doc.close()

        # update dataframe
        df.loc[idx, "batch_pdf_name"] = batch_pdf_name

    batch_doc.save(
        batch_pdf_path,
        garbage=4,
        deflate=True,
        clean=True
    )
    batch_doc.close()

print("Batch PDFs created.")

Batch PDFs created.
